### **Extracción de candidatos**
Carga ejemplares seleccionados del Boletín Oficial de la Ciudad de Buenos Aires,  extrae fragmentos de texto que contienen palabras clave y verbos que denotan una acción normativa, y los guarda en una archivo csv para su posterior etiquetado.

#### **Celda 1 — Imports y rutas**

In [1]:
import os, sys
from pathlib import Path
import pandas as pd

# Permitir importar módulos desde src/ (el notebook está dentro de notebooks/)
sys.path.append(os.path.abspath(".."))

from src.config import DATA_RAW, DATA_PROCESSED
from src.candidates import run_batch

DATA_RAW, DATA_PROCESSED

(WindowsPath('C:/Users/juand/Documents/GitHub/relevamiento-boletin-oficial-caba-con-llm/data/raw'),
 WindowsPath('C:/Users/juand/Documents/GitHub/relevamiento-boletin-oficial-caba-con-llm/data/processed'))

#### **Celda 2 — Configuración de ejecución**

In [2]:
INPUT_DIR  = DATA_RAW / "inbox"          
OUT_CSV    = DATA_PROCESSED / "candidatos.csv"
MASTER_CSV = DATA_PROCESSED / "dataset_master.csv"
ARCHIVE_DIR = DATA_RAW / "archive"      

print("Carpeta de PDFs (inbox):", INPUT_DIR)
print("Salida corrida:", OUT_CSV)
print("Maestro acumulado:", MASTER_CSV)
print("Carpeta de archivo:", ARCHIVE_DIR)

Carpeta de PDFs (inbox): C:\Users\juand\Documents\GitHub\relevamiento-boletin-oficial-caba-con-llm\data\raw\inbox
Salida corrida: C:\Users\juand\Documents\GitHub\relevamiento-boletin-oficial-caba-con-llm\data\processed\candidatos.csv
Maestro acumulado: C:\Users\juand\Documents\GitHub\relevamiento-boletin-oficial-caba-con-llm\data\processed\dataset_master.csv
Carpeta de archivo: C:\Users\juand\Documents\GitHub\relevamiento-boletin-oficial-caba-con-llm\data\raw\archive


#### **Celda 3 — Ejecutar el pipeline**

In [3]:
run_batch(
    input_dir=INPUT_DIR,
    out_csv=OUT_CSV,
    filter_has_action=False,
    master_csv=MASTER_CSV,
)
print("✅ Listo")

✅ Listo


#### **Celda 4 — Ver resultados de la corrida actual**

In [4]:
from pathlib import Path
import shutil

processed = list(INPUT_DIR.glob("*.pdf"))
for pdf in processed:
    shutil.move(str(pdf), str(ARCHIVE_DIR / pdf.name))

print(f"Movidos {len(processed)} PDFs de inbox → archive.")

df = pd.read_csv(OUT_CSV) if OUT_CSV.exists() else pd.DataFrame()
print("Filas en candidatos.csv:", len(df))
df.head(10)

Movidos 6 PDFs de inbox → archive.
Filas en candidatos.csv: 171


,pdf,pdf_path,keyword,keyword_is_regex,start,end,context,context_hash,has_action,action_hits
0,20180102.pdf,C:\Users\juand\Documents\GitHub\relevamiento-b...,Ley Tarifaria,0,93400,93413,posibles situaciones de emergencias. g) Descri...,c897777f493aaf0f9a11e9cb98f1ad08,1,regex:\bderogase\b;regex:\bestablecer\b;regex:...
1,20180102.pdf,C:\Users\juand\Documents\GitHub\relevamiento-b...,Ley Tarifaria,0,93482,93495,prevenir y controlar los riesgos sobre las per...,096e1256e8d1ab097b43d3d025f8e9c1,1,regex:\bderogase\b;regex:\bestablecer\b;regex:...
2,20180102.pdf,C:\Users\juand\Documents\GitHub\relevamiento-b...,Ley Tarifaria,0,1219502,1219515,las costas judiciales. Articulo 8deg.- Registr...,103f747eca8650550b2e81355c327869,1,regex:\bestablecer\b;regex:\bdeterminar\b
3,20180102.pdf,C:\Users\juand\Documents\GitHub\relevamiento-b...,Catastro,0,402531,402539,"CENTAVOS (4.043.906,25) a valores Basicos equi...",569e6a83af08e202c45cc977924b4cf7,1,regex:\bapruebase\b;regex:\bestablecese\b
4,20180102.pdf,C:\Users\juand\Documents\GitHub\relevamiento-b...,Catastro,0,421570,421578,"propias, No 5285 - 02/01/2018 Boletin Oficial ...",12b279248109b6e101533561d1557582,1,regex:\baprueba\b;regex:\bapruebase\b;regex:\b...
5,20180102.pdf,C:\Users\juand\Documents\GitHub\relevamiento-b...,Catastro,0,808832,808840,"(a NPT), mas servicios, alcanzando una altura ...",2f5b86058af3b85ed64e8ef4c583bd01,1,regex:\bdeterminar\b
6,20180102.pdf,C:\Users\juand\Documents\GitHub\relevamiento-b...,Catastro,0,829365,829373,"desde la LO de la Av. Varela, segun lo estable...",a5394743c658dc55ff56df391e1e2e85,0,NaN
7,20180118.pdf,C:\Users\juand\Documents\GitHub\relevamiento-b...,Ley Tarifaria,0,65543,65556,por el plazo del financiamiento hasta su cance...,ba07c160fe9466f4c5c70e2f38926255,1,regex:\bestablece\b;regex:\bincorporar\b
8,20180118.pdf,C:\Users\juand\Documents\GitHub\relevamiento-b...,Ley Tarifaria,0,65878,65891,"a las obligaciones asumidas, de acuerdo al cro...",dbaed90f14b2e822050305077ee8c8e7,1,regex:\bestablece\b;regex:\bincorporar\b
9,20180118.pdf,C:\Users\juand\Documents\GitHub\relevamiento-b...,Ley Tarifaria,0,601689,601702,"y demas efectos, pase a la Unidad de Compras y...",4fdf031a11dc43e63541e3e40a591a49,1,regex:\botorgase\b;regex:\brectificase\b


#### **Celda 5 — Ver estado del maestro (acumulado)**

In [5]:
dm = pd.read_csv(MASTER_CSV) if MASTER_CSV.exists() else pd.DataFrame()
print("Filas en dataset_master.csv:", len(dm))
if len(dm):
    display(dm.sample(min(10, len(dm))))
    print("\nBalance por has_action:")
    print(dm["has_action"].value_counts(dropna=False))

Filas en dataset_master.csv: 342


,pdf,pdf_path,keyword,keyword_is_regex,start,end,context,context_hash,has_action,action_hits
135,20180206.pdf,C:\Users\juand\Documents\GitHub\relevamiento-b...,Catastro,0,162913,162921,de planos otorgado por la Autoridad de Aplicac...,16661ab704ff8a6599d1af6f7fb8f27f,0,NaN
10,20190314.pdf,C:\Users\juand\Documents\GitHub\relevamiento-b...,Catastro,0,318268,318276,"la calle Bernardo de Irigoyen, entre Avenida S...",b4c3bd1c06302157479303a0a7e003ca,0,NaN
4,20190116.pdf,C:\Users\juand\Documents\GitHub\relevamiento-b...,Catastro,0,730269,730277,actuaciones administrativas; Que en el Anexo I...,306252af93baf43b5e1b00bae2c68000,0,NaN
221,20180206.pdf,C:\Users\juand\Documents\GitHub\relevamiento-b...,Impacto ambiental,0,605873,605890,Aptitud Ambiental correspondiente al solicitan...,e4b01088f79cb229e13fe022f71eb859,1,regex:\bestablecer\b;regex:\bestablecese\b;reg...
121,20180206.pdf,C:\Users\juand\Documents\GitHub\relevamiento-b...,Ley Tarifaria,0,579651,579664,"electrico, para aquellos productos que lo nece...",d311eeef5350f24ea1251f030d76b86e,0,NaN
279,20180206.pdf,C:\Users\juand\Documents\GitHub\relevamiento-b...,Catastro,0,177246,177254,el tramite de regularizacion. Que la finca en ...,83f9d74ad73f654632f0f41578dcde26,0,NaN
222,20180206.pdf,C:\Users\juand\Documents\GitHub\relevamiento-b...,Impacto ambiental,0,610593,610610,de la actividad al cumplimiento de toda norma ...,2447d23d828826292f99d822838bd252,1,regex:\bestablecer\b;regex:\bestablecese\b;reg...
63,20180206.pdf,C:\Users\juand\Documents\GitHub\relevamiento-b...,Impacto ambiental,0,531917,531934,retiro de la presente Disposicion. Articulo 8d...,f73769675d9da05c4ea7c062483ca8d7,0,NaN
319,20190314.pdf,C:\Users\juand\Documents\GitHub\relevamiento-b...,Catastro,0,320791,320799,"No 2019 07243586- GCBA-SSREGIC, y CONSIDERANDO...",ffc3bd3831684b97bc65889bc83f7ad2,1,regex:\baprueba\b;regex:\bapruebase\b;regex:\b...
194,20180206.pdf,C:\Users\juand\Documents\GitHub\relevamiento-b...,Impacto ambiental,0,523483,523500,"Comuna 14, de la Ciudad Autonoma de Buenos Air...",5ae5102d8061a3779a3d5cf6456aadad,1,regex:\bdejar sin efecto\b;regex:\bdejese sin ...



Balance por has_action:
has_action
0    193
1    149
Name: count, dtype: int64
